# Práctica guiada: del modelo legado al artefacto explícito

En la semana 3 usamos un payload **joblib** con tres claves: el estimador, los nombres de las features y la versión del modelo. En esta práctica vamos a separar responsabilidades:

- **model.joblib**: el objeto ejecutable;
- **manifest.json**: el contrato que permite saber si el objeto es compatible;
- código y pruebas: las reglas de validación y el comportamiento ante errores.

## Resultado esperado

Al terminar tendrás un manifiesto mínimo escrito como JSON, una lista de invariantes y una matriz de fallos que otra persona podrá implementar en la segunda clase.

## 1. Inspeccionar el punto de partida

Primero comprueba qué columnas recibe el módulo de inferencia de la semana 3. No cambies todavía el modelo ni ejecutes predicciones.

In [1]:
from pathlib import Path
import csv

week_root = next(
    parent
    for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root / "assets/03-wine-quality/inference_samples.csv"
if not input_path.exists():
    input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"

with input_path.open(newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    rows = list(reader)
print(f"Fichero: {input_path}")
print(f"Columnas: {reader.fieldnames}")
print(f"Filas de ejemplo: {len(rows)}")

Fichero: C:\Users\Byadr\Documents\Master\operacion_modelos\icai-muiaap-operacion-de-modelos\semana3\assets\03-wine-quality\inference_samples.csv
Columnas: ['sample_id', 'fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
Filas de ejemplo: 5


## 2. Diseñar la frontera del artefacto

Completa esta tabla antes de escribir código. La pregunta clave es: **¿qué necesita un proceso de inferencia para decidir que puede utilizar el artefacto?**

| Elemento | ¿Dónde vive? | ¿Qué riesgo evita? |
| --- | --- | --- |
| Estimador entrenado | **model.joblib** | Es el objeto que hace predict y predict_proba. Solo se puede guardar en binario y no hace falta abrirlo para saber si es compatible. |
| Versión del esquema | manifest.json | Que el cargador lea un manifiesto con otra forma (otros campos) sin darse cuenta. |
| Orden y nombres de features | manifest.json (y FEATURE_NAMES en el código para comparar) | Que el modelo reciba las once medidas en otro orden y prediga mal sin dar ningún error. |
| Versión del preprocesado | manifest.json | Usar el modelo con una transformación distinta a la que se usó al entrenarlo. |
| Etiquetas de salida permitidas | manifest.json | Que la respuesta devuelva una categoría que quien consume el modelo no conoce. |
| Versión del modelo | manifest.json | No saber qué modelo ha generado cada predicción. |
| Tipo de estimador | manifest.json | Cargar un objeto de otra clase que no tiene la interfaz que esperamos. |
| Reglas de validación | código (artifact.py y tests) | Que las comprobaciones dependan del notebook. En el código se pueden probar con tests y son iguales para todos los bundles. |

## 3. Práctica 2 — Construir un manifiesto mínimo

Completa el borrador de Python de la siguiente celda y ejecútalo para producir un JSON legible. Mantén los nombres de campo estables: el manifiesto es una interfaz entre quien empaqueta el modelo y quien lo ejecuta.

~~~json
{
  "schema_version": "",
  "model_version": "",
  "preprocessing_version": "",
  "feature_names": [],
  "output_labels": [],
  "estimator_type": ""
}
~~~

Comprueba antes de continuar que:

- aparecen exactamente las seis claves obligatorias;
- `feature_names` tiene once nombres y no incluye `sample_id`;
- `output_labels` representa las categorías que la interfaz podrá devolver.

Decide también qué debe pasar si:

1. falta **manifest.json**;
2. aparecen features desconocidas o en un orden distinto;
3. el modelo devuelve una etiqueta que no está en **output_labels**;
4. una fila del CSV no cumple el contrato de entrada.

In [2]:
import json

feature_names = [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol",
]

# No modifiques este orden: procede del contrato de la semana 3.
legacy_model_version = "wine-quality-rf-v1"

manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",
    "model_version": legacy_model_version,
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],
    "estimator_type": "RandomForestClassifier",
}

required_keys = {
    "schema_version",
    "model_version",
    "preprocessing_version",
    "feature_names",
    "output_labels",
    "estimator_type",
}

assert set(manifest_draft) == required_keys
assert len(manifest_draft["feature_names"]) == 11
assert "sample_id" not in manifest_draft["feature_names"]

print(json.dumps(manifest_draft, indent=2, ensure_ascii=False))
print("\nLa estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.")

{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "RandomForestClassifier"
}

La estructura es válida. Sustituye los TODO y las etiquetas provisionales por decisiones justificadas.


**Respuestas**

Valores elegidos:

- schema_version = wine-quality-bundle-v1, es la primera versión de la forma del manifiesto.
- model_version = wine-quality-rf-v1, es la versión que trae el payload de la semana 3.
- preprocessing_version = wine-red-features-v1, es la versión de preprocess.py de la semana 3. No es lo mismo que model_version: puedo reentrenar el modelo sin cambiar el preprocesado y al revés.
- output_labels = needs_review, acceptable y excellent, que son las categorías del contrato de salida de la semana 3 (low, medium y high no existen en ese contrato).
- estimator_type = RandomForestClassifier, que es la clase del estimador del payload.

Qué pasa si:

1. Falta manifest.json: la carga se para con un error de fichero no encontrado y no se llega a abrir model.joblib.
2. Hay features desconocidas o en otro orden: se rechaza al validar el manifiesto, porque feature_names tiene que ser igual que FEATURE_NAMES y en el mismo orden. Si la columna desconocida viene en el CSV, la rechaza el contrato de entrada.
3. El modelo devuelve una etiqueta que no está en output_labels: se rechaza al validar la salida y esa predicción no se escribe.
4. Una fila no cumple el contrato de entrada: falla WineQualityRequest, el error dice la fila y el sample_id y no se escribe el CSV.

Invariantes que deben bloquear la carga o la inferencia:

1. feature_names del manifiesto es exactamente FEATURE_NAMES de la semana 3 (mismos once nombres y mismo orden).
2. schema_version y preprocessing_version son las que conoce el código (wine-quality-bundle-v1 y wine-red-features-v1).
3. El estimador cargado tiene predict y predict_proba y su clase coincide con estimator_type.
4. Cada predicción tiene una etiqueta de output_labels y una confianza entre 0 y 1.

## 4. Práctica 3 — Anticipar fallos

Antes de ver la demo del profesorado, completa la tabla. No describas solo el error: indica en qué frontera debería detenerse el flujo.

| Cambio | Frontera que debe rechazarlo | Mensaje útil esperado | ¿Se escribe CSV? |
| --- | --- | --- | --- |
| Se intercambian `density` y `alcohol` | Carga del bundle, al validar el manifiesto antes de joblib.load | feature_names no coincide con el orden de la semana 3 | No |
| Cambia `preprocessing_version` | Carga del bundle, al validar el manifiesto antes de joblib.load | preprocessing_version incompatible: se esperaba wine-red-features-v1 | No |
| El modelo devuelve `unknown` | Validación de la salida, después de predict | el modelo ha devuelto la etiqueta unknown, que no está en output_labels | No |
| La petición incluye `ph = 99` | Contrato de entrada (WineQualityRequest), antes de preprocesar | fila 2 (sample_id=red-001): ph debe ser menor o igual que 4.5 | No |
| Cambia estimator_type (por ejemplo a LogisticRegression) | Carga del bundle, justo después de joblib.load y antes de inferir | el estimador es RandomForestClassifier pero el manifiesto dice LogisticRegression | No |

## 5. Práctica 4 — Preparar la implementación de la segunda clase

Antes de pasar al taller, deja cerradas estas decisiones:

- ¿Qué campos son obligatorios y cuáles tienen valores por defecto?
- ¿Aceptarás campos desconocidos en el manifiesto?
- ¿Validarás el manifiesto antes de cargar el objeto serializado?
- ¿El CSV de salida se escribe fila a fila o solo después de validar todas las entradas?
- ¿Qué mensaje verá la persona que ejecuta la CLI cuando el contrato falle?
- ¿Qué ocurre si una segunda fila es inválida después de haber procesado una primera fila correcta?

La segunda clase convertirá estas decisiones en **ArtifactManifest**, **save_model_bundle**, **load_model_bundle**, inferencia y pruebas.

**Respuestas**

- Los seis campos del manifiesto son obligatorios y ninguno tiene valor por defecto. Si falta uno no se puede comprobar la compatibilidad.
- No acepto campos desconocidos (extra="forbid"). Un campo raro suele indicar que el manifiesto es de otra versión.
- Sí, primero se lee y valida manifest.json y solo si es correcto se hace joblib.load. Aun así, que joblib.load funcione no hace el binario seguro: solo cargo bundles de una fuente de confianza.
- El CSV de salida se escribe solo después de validar e inferir todas las filas. Mientras tanto las predicciones se guardan en memoria.
- La persona que ejecuta la CLI ve un mensaje con la fila, el sample_id y el campo que falla (o el campo del manifiesto que no coincide), y el comando termina con código 2.
- Si la segunda fila es inválida se para todo y no se escribe ningún CSV, aunque la primera fila estuviera bien. No se publica una salida parcial porque alguien podría usar un CSV incompleto pensando que está entero y sin saber qué filas faltan. Es todo o nada: escribo en un fichero temporal y al final lo renombro.